# Solving the Beam Equation: Numerical Methods vs. Neural Networks

This notebook solves the **1-D Euler–Bernoulli beam equation** using two approaches:

1. **Numerical Methods** — Explicit Central Difference (5-point biharmonic stencil) and Crank–Nicolson-style Implicit Scheme (pentadiagonal)
2. **Neural Network** — Physics-Informed Neural Network (PINN) via PyTorch

---

## Problem Statement

The Euler–Bernoulli beam equation governs **transverse vibrations** of a thin elastic beam:

$$\frac{\partial^2 u}{\partial t^2} + \gamma^2\,\frac{\partial^4 u}{\partial x^4} = 0, \qquad x \in [0, L],\; t \in [0, T]$$

where $\gamma^2 = EI/(\rho A)$ is the **flexural rigidity coefficient** — $E$ is Young's modulus, $I$ the second moment of area, $\rho$ the density, and $A$ the cross-section area.

Key properties:
- **Fourth-order spatial derivative** — the highest-order PDE in this notebook collection, arising from bending stiffness rather than tension.
- **Dispersive, not diffusive** — waves of different wavelengths travel at different speeds, but **no amplitude decay**. Energy $E = \tfrac{1}{2}\int_0^L[u_t^2 + \gamma^2 u_{xx}^2]\,dx$ is conserved exactly.
- **Dispersion relation**: substituting $u = e^{i(kx - \omega t)}$ gives $\omega = \gamma k^2$. The **phase velocity** $v_p = \omega/k = \gamma k$ and **group velocity** $v_g = d\omega/dk = 2\gamma k$ both increase with $k$ — higher-frequency bending waves travel faster.
- **Quadratic dispersion** ($\omega \propto k^2$) vs. *linear* for the wave equation ($\omega \propto k$) and *cubic* for Airy ($\omega \propto k^3$).

**Boundary conditions** — simply-supported (pinned-pinned):

$$u(0, t) = u(L, t) = 0, \qquad u_{xx}(0, t) = u_{xx}(L, t) = 0$$

The first condition pins the ends; the second enforces zero bending moment.

**Initial conditions** — first mode, released from rest:

$$u(x, 0) = \sin\!\left(\frac{\pi x}{L}\right), \qquad \frac{\partial u}{\partial t}(x, 0) = 0$$

**Exact solution** — the $n$-th mode eigenfrequency is $\omega_n = \gamma(n\pi/L)^2$. For the first mode with $u_t(x,0) = 0$:

$$u^*(x, t) = \sin\!\left(\frac{\pi x}{L}\right)\cos(\omega_1\,t), \qquad \omega_1 = \gamma\!\left(\frac{\pi}{L}\right)^{\!2}$$

| Mode $n$ | $k_n = n\pi/L$ | $\omega_n = \gamma k_n^2$ | Ratio $\omega_n/\omega_1$ |
|----------|---------------|-------------------------|-------------------------|
| 1 | $\pi/L$ | $\gamma(\pi/L)^2$ | 1 |
| 2 | $2\pi/L$ | $4\gamma(\pi/L)^2$ | 4 |
| 3 | $3\pi/L$ | $9\gamma(\pi/L)^2$ | 9 |

The eigenfrequencies grow as $n^2$ — the beam favours low-frequency bending modes; high modes oscillate extremely fast, which is why explicit numerical schemes face a **severe CFL constraint**: $\Delta t \propto \Delta x^2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.linalg import solve_banded
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
CMAP = "inferno"

# ---- Problem parameters ---------------------
GAMMA   = 1.0       # flexural wave-speed parameter  (γ² = EI/ρA)
L_DOM   = 1.0       # beam length [0, L]
T_END   = 1.0       # final time

# Derived quantities
K1      = np.pi / L_DOM                            # first-mode wavenumber
OMEGA1  = GAMMA * K1**2                             # first-mode eigenfrequency
T_PER   = 2 * np.pi / OMEGA1                       # first-mode period

In [ ]:
def u_init(x):
    """Initial displacement: first mode sin(πx/L)."""
    return np.sin(K1 * x)


def v_init(x):
    """Initial velocity: released from rest."""
    return np.zeros_like(x)


def u_exact(x, t):
    """Exact solution: first mode oscillating at ω₁."""
    return np.sin(K1 * x) * np.cos(OMEGA1 * t)


# Quick preview — exact solution at several times
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
x_plot = np.linspace(0, L_DOM, 500)
for ax, t_snap in zip(axes, [0.0, T_END / 3, 2 * T_END / 3, T_END]):
    ax.plot(x_plot, u_exact(x_plot, t_snap), "k-", lw=2)
    ax.fill_between(x_plot, u_exact(x_plot, t_snap), alpha=0.25, color="steelblue")
    ax.set_title(f"$t = {t_snap:.3f}$")
    ax.set_xlabel("$x$")
    ax.set_ylabel("$u$")
    ax.set_ylim(-1.3, 1.3)
    ax.grid(alpha=0.3)

plt.suptitle("Exact Solution — Beam Equation (Simply-Supported First Mode)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"γ = {GAMMA},  L = {L_DOM},  T = {T_END}")
print(f"ω₁ = γ(π/L)² = {OMEGA1:.4f}")
print(f"Period T₁ = 2π/ω₁ = {T_PER:.4f}")
print(f"T_end / T₁ = {T_END / T_PER:.2f} periods")
print(f"Dispersion: ω_n = γ(nπ/L)² — mode 2 oscillates 4× faster than mode 1")

---

## Part 1 — Numerical Methods

### 1-A. Explicit Central Difference

Central differences in time for $u_{tt}$ and the standard **5-point biharmonic stencil** for $u_{xxxx}$:

$$\frac{u_j^{n+1} - 2u_j^n + u_j^{n-1}}{\Delta t^2} + \gamma^2\,\frac{u_{j-2} - 4u_{j-1} + 6u_j - 4u_{j+1} + u_{j+2}}{\Delta x^4} = 0$$

Solving for $u_j^{n+1}$:

$$u_j^{n+1} = 2u_j^n - u_j^{n-1} - \mu\,(u_{j-2}^n - 4u_{j-1}^n + 6u_j^n - 4u_{j+1}^n + u_{j+2}^n)$$

where $\mu = \gamma^2 \Delta t^2 / \Delta x^4$.

**Simply-supported ghost points**: at $x = 0$ we have $u(0) = 0$ and $u_{xx}(0) = 0$. The latter, combined with $u_0 = 0$, gives the ghost value $u_{-1} = -u_1$. Similarly at $x = L$: $u_{N+2} = -u_N$. This modifies the stencil at the first and last interior points.

**Stability** (von Neumann): $\mu \leq \tfrac{1}{4}$, i.e. $\gamma\,\Delta t / \Delta x^2 \leq \tfrac{1}{2}$.

The **quadratic** CFL dependence on $\Delta x$ makes explicit time-stepping **very expensive** for the beam equation — far more restrictive than the wave equation's $\Delta t \propto \Delta x$.

**Bootstrap** (with $u_t(x,0) = 0$): Taylor expansion gives

$$u_j^1 = u_j^0 - \frac{\gamma^2 \Delta t^2}{2}\,\frac{u_{xxxx}^0}{1} = u_j^0 - \frac{\mu}{2}\,D^4_h u_j^0$$

In [ ]:
def D4_apply(u_int):
    """
    Apply biharmonic stencil [1, -4, 6, -4, 1] to interior values
    with simply-supported ghost points: u_{-1} = -u_1, u_{N+2} = -u_N.
    """
    u_ext = np.concatenate([[-u_int[0]], [0.0], u_int, [0.0], [-u_int[-1]]])
    return (u_ext[:-4] - 4 * u_ext[1:-3] + 6 * u_ext[2:-2]
            - 4 * u_ext[3:-1] + u_ext[4:])


def solve_explicit(Nx=100, T=T_END):
    """
    Explicit central-difference scheme for the beam equation.

    Parameters
    ----------
    Nx : int   Interior grid points.
    T  : float Final time.

    Returns
    -------
    x      : ndarray (Nx+2,)   Grid including boundary points.
    u_hist : list of ndarray   Solution snapshots.
    t_hist : list of float     Snapshot times.
    """
    x   = np.linspace(0, L_DOM, Nx + 2)
    dx  = x[1] - x[0]

    # CFL: μ = γ²Δt²/dx⁴ ≤ 1/4  →  Δt ≤ dx² / (2γ)
    dt_max = 0.45 * dx**2 / GAMMA            # safety factor < 0.5
    Nt     = max(int(np.ceil(T / dt_max)), 100)
    dt     = T / Nt
    mu     = GAMMA**2 * dt**2 / dx**4

    print(f"Explicit — Nx={Nx}, dx={dx:.5f}, Nt={Nt}, dt={dt:.2e}, "
          f"μ=γ²Δt²/dx⁴={mu:.4f}  (≤0.25 for stability)")

    # Level n=0
    u_prev      = u_init(x).copy()
    u_prev[0]   = u_prev[-1] = 0.0

    # Level n=1 via Taylor (v_init = 0)
    u_curr       = u_prev.copy()
    u_curr[1:-1] = u_prev[1:-1] - (mu / 2) * D4_apply(u_prev[1:-1])
    u_curr[0] = u_curr[-1] = 0.0

    snap_steps = [0, Nt // 3, 2 * Nt // 3, Nt]
    u_hist, t_hist = [], []

    for n in range(Nt + 1):
        if n in snap_steps:
            u_snap = u_prev if n == 0 else u_curr
            u_hist.append(u_snap.copy())
            t_hist.append(n * dt)
        if n <= 0 or n == Nt:
            if n == Nt:
                break
            continue

        # Explicit update: u^{n+1} = 2u^n − u^{n−1} − μ D4(u^n)
        d4u          = D4_apply(u_curr[1:-1])
        u_next       = np.zeros_like(u_curr)
        u_next[1:-1] = 2 * u_curr[1:-1] - u_prev[1:-1] - mu * d4u
        u_next[0] = u_next[-1] = 0.0

        u_prev = u_curr
        u_curr = u_next

    return x, u_hist, t_hist


x_ex, u_ex_hist, t_ex_hist = solve_explicit(Nx=100)

# Error at final time
u_ref_ex   = u_exact(x_ex, T_END)
err_ex     = np.abs(u_ex_hist[-1] - u_ref_ex)
print(f"Max absolute error  : {err_ex.max():.3e}")
print(f"Mean absolute error : {err_ex.mean():.3e}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (u_snap, t_snap) in enumerate(zip(u_ex_hist, t_ex_hist)):
    u_ref = u_exact(x_ex, t_snap)

    axes[0, col].plot(x_ex, u_ref, "k-", lw=2, label="Exact")
    axes[0, col].plot(x_ex, u_snap, "b--", lw=1.5, label="Explicit")
    axes[0, col].set_title(f"$t = {t_snap:.3f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.3, 1.3)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(u_snap - u_ref)
    axes[1, col].plot(x_ex, err, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Explicit Central Difference (Biharmonic Stencil) — Beam Equation",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 1-B. Implicit (Crank–Nicolson-style) Scheme

Time-averaging the biharmonic operator over three levels gives an unconditionally stable implicit scheme:

$$\frac{u^{n+1} - 2u^n + u^{n-1}}{\Delta t^2} + \frac{\gamma^2}{4\,\Delta x^4}\left(D^4_h u^{n+1} + 2\,D^4_h u^n + D^4_h u^{n-1}\right) = 0$$

With $\sigma = \gamma^2 \Delta t^2 / (4\,\Delta x^4)$:

$$(I + \sigma B_4)\,\mathbf{u}^{n+1} = 2\mathbf{u}^n - \mathbf{u}^{n-1} - \sigma\,(2\,D^4_h\mathbf{u}^n + D^4_h\mathbf{u}^{n-1})$$

where $B_4$ is the **pentadiagonal** biharmonic matrix (bandwidth 2) with simply-supported modifications at the corners:

| Band | Coefficient |
|------|------------|
| 2nd sub/super | $\sigma$ |
| 1st sub/super | $-4\sigma$ |
| diagonal (interior) | $1 + 6\sigma$ |
| diagonal (first/last row) | $1 + 5\sigma$ |

The corner modification ($5\sigma$ instead of $6\sigma$) comes from the simply-supported ghost-point condition $u_{-1} = -u_1$, $u_{N+2} = -u_N$.

This scheme is **unconditionally stable** and $\mathcal{O}(\Delta t^2, \Delta x^2)$ — no CFL constraint, at the cost of a pentadiagonal solve per time step.

In [ ]:
def solve_implicit(Nx=200, Nt=500, T=T_END):
    """
    Implicit (CN-style) scheme for the beam equation.

    Parameters
    ----------
    Nx : int   Interior grid points.
    Nt : int   Number of time steps.

    Returns
    -------
    x      : ndarray (Nx+2,)
    u_hist : list of ndarray
    t_hist : list of float
    """
    x   = np.linspace(0, L_DOM, Nx + 2)
    dx  = x[1] - x[0]
    dt  = T / Nt
    N   = Nx

    sigma = GAMMA**2 * dt**2 / (4 * dx**4)
    print(f"Implicit — Nx={Nx}, dx={dx:.5f}, Nt={Nt}, dt={dt:.5f}, σ={sigma:.4e}")

    # Pentadiagonal LHS band matrix  (I + σ B4)
    A_band          = np.zeros((5, N))
    A_band[0, 2:]   = sigma                              # 2nd super-diagonal
    A_band[1, 1:]   = -4 * sigma                         # 1st super-diagonal
    A_band[2, :]    = 1 + 6 * sigma                      # main diagonal
    A_band[2, 0]    = 1 + 5 * sigma                      # corner: simply-supported
    A_band[2, -1]   = 1 + 5 * sigma                      # corner: simply-supported
    A_band[3, :-1]  = -4 * sigma                         # 1st sub-diagonal
    A_band[4, :-2]  = sigma                              # 2nd sub-diagonal

    # Initialise: u^0 and bootstrap u^1
    u_prev      = u_init(x).copy()
    u_prev[0]   = u_prev[-1] = 0.0

    mu_boot      = GAMMA**2 * dt**2 / dx**4
    u_curr       = u_prev.copy()
    u_curr[1:-1] = u_prev[1:-1] - (mu_boot / 2) * D4_apply(u_prev[1:-1])
    u_curr[0] = u_curr[-1] = 0.0

    snap_steps = [0, Nt // 3, 2 * Nt // 3, Nt]
    u_hist, t_hist = [], []

    for n in range(Nt + 1):
        if n in snap_steps:
            u_snap = u_prev if n == 0 else u_curr
            u_hist.append(u_snap.copy())
            t_hist.append(n * dt)
        if n <= 0 or n == Nt:
            if n == Nt:
                break
            continue

        # RHS = 2u^n − u^{n-1} − σ(2·D4(u^n) + D4(u^{n-1}))
        d4_curr = D4_apply(u_curr[1:-1])
        d4_prev = D4_apply(u_prev[1:-1])
        rhs = (2 * u_curr[1:-1] - u_prev[1:-1]
               - sigma * (2 * d4_curr + d4_prev))

        u_next       = np.zeros_like(u_curr)
        u_next[1:-1] = solve_banded((2, 2), A_band, rhs)
        u_next[0]    = 0.0
        u_next[-1]   = 0.0

        u_prev = u_curr
        u_curr = u_next

    return x, u_hist, t_hist


x_im, u_im_hist, t_im_hist = solve_implicit(Nx=200, Nt=500)

u_ref_im = u_exact(x_im, T_END)
err_im   = np.abs(u_im_hist[-1] - u_ref_im)
print(f"Max absolute error  : {err_im.max():.3e}")
print(f"Mean absolute error : {err_im.mean():.3e}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (u_snap, t_snap) in enumerate(zip(u_im_hist, t_im_hist)):
    u_ref = u_exact(x_im, t_snap)

    axes[0, col].plot(x_im, u_ref, "k-", lw=2, label="Exact")
    axes[0, col].plot(x_im, u_snap, "g--", lw=1.5, label="Implicit")
    axes[0, col].set_title(f"$t = {t_snap:.3f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.3, 1.3)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(u_snap - u_ref)
    axes[1, col].plot(x_im, err, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Implicit (CN-style, Pentadiagonal) — Beam Equation",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Grid-refinement convergence study ----------------------------------------
Nx_list = [25, 50, 100, 200]
err_ex_conv, err_im_conv = [], []

for Nx in Nx_list:
    # Explicit (CFL-limited, Nt computed automatically)
    x_c, u_c, t_c = solve_explicit(Nx, T=T_END)
    u_ex_c         = u_exact(x_c, t_c[-1])
    err_ex_conv.append(np.max(np.abs(u_c[-1] - u_ex_c)))

    # Implicit — large fixed Nt to ensure temporal error is subdominant
    x_c2, u_c2, t_c2 = solve_implicit(Nx, Nt=5000, T=T_END)
    u_ex_c2           = u_exact(x_c2, t_c2[-1])
    err_im_conv.append(np.max(np.abs(u_c2[-1] - u_ex_c2)))

dh = [L_DOM / (Nx + 1) for Nx in Nx_list]

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(dh, err_ex_conv, "bs-", lw=1.5, label="Explicit")
ax.loglog(dh, err_im_conv, "ro-", lw=1.5, label="Implicit (CN-style)")
ax.loglog(dh, [h**2 * err_ex_conv[0] / dh[0]**2 for h in dh],
          "b--", lw=1.0, label="$\\mathcal{O}(h^2)$")
ax.loglog(dh, [h**2 * err_im_conv[0] / dh[0]**2 for h in dh],
          "r--", lw=1.0, label="$\\mathcal{O}(h^2)$ (ref)")
ax.set_xlabel("Grid spacing $h$")
ax.set_ylabel("Max absolute error at $t = T$")
ax.set_title("Convergence Study — FDM Schemes")
ax.legend()
ax.grid(which="both", alpha=0.3)
plt.tight_layout()
plt.show()

---

## Part 2 — Neural Network: Physics-Informed Neural Network (PINN)

### How It Works

A PINN learns $\hat{u}_\theta(x, t)$ by minimising:

$$\mathcal{L} = \underbrace{\mathcal{L}_{IC}}_{\text{initial conditions}} + \underbrace{\mathcal{L}_{BC}}_{\text{boundary}} + \underbrace{\mathcal{L}_{PDE}}_{\text{physics residual}}$$

$$\mathcal{L}_{IC} = \frac{1}{N_{IC}}\sum_k \Bigl(\hat{u}(x_k, 0) - u_0(x_k)\Bigr)^2 + \frac{1}{N_{IC}}\sum_k \Bigl(\hat{u}_t(x_k, 0)\Bigr)^2$$

$$\mathcal{L}_{BC} = \frac{1}{N_{BC}}\sum_k \left[\hat{u}(0, t_k)^2 + \hat{u}(L, t_k)^2 + \hat{u}_{xx}(0, t_k)^2 + \hat{u}_{xx}(L, t_k)^2\right]$$

Note: the simply-supported BCs require enforcing **both** $u = 0$ and $u_{xx} = 0$ at each end — four constraints total.

$$\mathcal{L}_{PDE} = \frac{1}{N_f}\sum_k \left(\frac{\partial^2\hat{u}}{\partial t^2} + \gamma^2\,\frac{\partial^4\hat{u}}{\partial x^4}\right)^2$$

The PDE residual requires the **fourth spatial derivative** $u_{xxxx}$, computed by **four successive autograd passes**:

$$u \xrightarrow{\partial_x} u_x \xrightarrow{\partial_x} u_{xx} \xrightarrow{\partial_x} u_{xxx} \xrightarrow{\partial_x} u_{xxxx}$$

Tanh activations ensure all four derivatives exist and are smooth.

Training: **Adam** warm-up → **L-BFGS** fine-tuning.

In [ ]:
# -----------------------------------------------------------------
# Network Architecture
# -----------------------------------------------------------------
class BeamPINN(nn.Module):
    """Fully-connected PINN: (x, t) → u.
    Tanh activations ensure smooth fourth-order derivatives via autograd.
    """

    def __init__(self, hidden_layers=5, hidden_dim=80):
        super().__init__()
        layers = [nn.Linear(2, hidden_dim), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))


def grad1(u, v):
    return torch.autograd.grad(
        u, v, grad_outputs=torch.ones_like(u),
        create_graph=True, retain_graph=True
    )[0]


# -----------------------------------------------------------------
# Collocation / IC / BC Points
# -----------------------------------------------------------------
N_IC    = 2000    # IC points (displacement + velocity)
N_BC    = 1000    # boundary points (each side)
N_INT   = 10000   # interior PDE collocation points

# ---- IC at t = 0 ----------------------------------
x_ic = (torch.rand(N_IC, 1) * L_DOM).requires_grad_(True)
t_ic = torch.zeros(N_IC, 1, requires_grad=True)
u_ic = torch.tensor(
    u_init(x_ic.detach().numpy()), dtype=torch.float32
)

# ---- Boundary points (x = 0 and x = L) -----------
t_bc       = torch.rand(N_BC, 1) * T_END
x_bc_left  = torch.zeros(N_BC, 1, requires_grad=True)
x_bc_right = torch.full((N_BC, 1), L_DOM, requires_grad=True)

# ---- Interior PDE collocation points --------------
x_int = (torch.rand(N_INT, 1) * L_DOM).requires_grad_(True)
t_int = (torch.rand(N_INT, 1) * T_END).requires_grad_(True)

mse = nn.MSELoss()

print(f"IC  points : {N_IC}")
print(f"BC  points : {2 * N_BC} (u=0 and u_xx=0 at each end)")
print(f"PDE points : {N_INT}")

In [ ]:
def compute_loss(model):
    # ---- IC loss: displacement + velocity ------
    u_pred_ic   = model(x_ic, t_ic)
    loss_ic_u   = mse(u_pred_ic, u_ic)

    u_t_ic      = grad1(u_pred_ic, t_ic)
    loss_ic_v   = mse(u_t_ic, torch.zeros_like(u_t_ic))

    loss_ic     = loss_ic_u + loss_ic_v

    # ---- Boundary loss: u = 0 AND u_xx = 0 at x = 0, L -----
    u_left       = model(x_bc_left,  t_bc)
    u_right      = model(x_bc_right, t_bc)
    loss_bc_u    = mse(u_left,  torch.zeros_like(u_left)) + mse(u_right, torch.zeros_like(u_right))

    u_x_left     = grad1(u_left,   x_bc_left)
    u_xx_left    = grad1(u_x_left, x_bc_left)
    u_x_right    = grad1(u_right,  x_bc_right)
    u_xx_right   = grad1(u_x_right, x_bc_right)
    loss_bc_uxx  = mse(u_xx_left, torch.zeros_like(u_xx_left)) + mse(u_xx_right, torch.zeros_like(u_xx_right))

    loss_bc      = loss_bc_u + loss_bc_uxx

    # ---- PDE residual: u_tt + γ²·u_xxxx = 0 -----
    u_pred  = model(x_int, t_int)
    u_t     = grad1(u_pred, t_int)
    u_tt    = grad1(u_t,    t_int)
    u_x     = grad1(u_pred, x_int)
    u_xx    = grad1(u_x,    x_int)
    u_xxx   = grad1(u_xx,   x_int)
    u_xxxx  = grad1(u_xxx,  x_int)
    residual = u_tt + GAMMA**2 * u_xxxx
    loss_pde = mse(residual, torch.zeros_like(residual))

    return loss_ic + loss_bc + loss_pde, loss_ic, loss_bc, loss_pde


# ---- Phase 1: Adam --------------------------------
model = BeamPINN(hidden_layers=5, hidden_dim=80)
opt_adam = optim.Adam(model.parameters(), lr=1e-3)
ADAM_EP = 5000
history = []

for ep in range(1, ADAM_EP + 1):
    opt_adam.zero_grad()
    loss, l_ic, l_bc, l_pde = compute_loss(model)
    loss.backward()
    opt_adam.step()
    history.append(loss.item())
    if ep % 1000 == 0:
        print(f"[Adam] Ep {ep:5d} | Loss {loss.item():.5f} | "
              f"IC {l_ic.item():.5f} | BC {l_bc.item():.5f} | PDE {l_pde.item():.5f}")

print("Adam phase done.")

In [ ]:
# ---- Phase 2: L-BFGS ----
opt_lbfgs = optim.LBFGS(
    model.parameters(),
    max_iter=500, tolerance_grad=1e-9, tolerance_change=1e-12,
    history_size=50, line_search_fn="strong_wolfe"
)


def closure():
    opt_lbfgs.zero_grad()
    loss, _, _, _ = compute_loss(model)
    loss.backward()
    history.append(loss.item())
    return loss


opt_lbfgs.step(closure)
final, _, _, _ = compute_loss(model)
print(f"L-BFGS done.  Final loss: {final.item():.6f}")

# Training loss curve
fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogy(history, color="steelblue", lw=1.0)
ax.axvline(x=ADAM_EP, color="tomato", ls="--", lw=1.5, label="Adam → L-BFGS")
ax.set_xlabel("Iteration")
ax.set_ylabel("Total Loss (log scale)")
ax.set_title("PINN Training Loss — Beam Equation")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Evaluate PINN on a dense x grid at each snapshot time ----
model.eval()
Nev  = 500
x_ev = np.linspace(0, L_DOM, Nev)


def pinn_predict(t_val):
    xt = torch.tensor(x_ev, dtype=torch.float32).unsqueeze(1)
    tt = torch.full((Nev, 1), t_val, dtype=torch.float32)
    with torch.no_grad():
        return model(xt, tt).numpy().ravel()


snap_times_pinn = [0.0, T_END / 3, 2 * T_END / 3, T_END]
U_pinn_snaps    = [pinn_predict(t) for t in snap_times_pinn]

# PINN error at final time
U_ex_pinn   = u_exact(x_ev, T_END)
err_pinn    = np.abs(U_pinn_snaps[-1] - U_ex_pinn)

fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for col, (U_snap, t_snap) in enumerate(zip(U_pinn_snaps, snap_times_pinn)):
    u_ref = u_exact(x_ev, t_snap)

    axes[0, col].plot(x_ev, u_ref, "k-", lw=2, label="Exact")
    axes[0, col].plot(x_ev, U_snap, "m--", lw=1.5, label="PINN")
    axes[0, col].set_title(f"$t = {t_snap:.3f}$")
    axes[0, col].set_xlabel("$x$")
    axes[0, col].set_ylabel("$u$")
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_ylim(-1.3, 1.3)
    axes[0, col].grid(alpha=0.3)

    err = np.abs(U_snap - u_ref)
    axes[1, col].plot(x_ev, err, "r-", lw=1.5)
    axes[1, col].set_title(f"Error  (max {err.max():.2e})")
    axes[1, col].set_xlabel("$x$")
    axes[1, col].set_ylabel("|error|")
    axes[1, col].grid(alpha=0.3)

plt.suptitle("Physics-Informed Neural Network — Beam Equation",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"PINN — Max error at t=T: {err_pinn.max():.2e}  |  Mean error: {err_pinn.mean():.2e}")

---

## Part 3 — Side-by-Side Comparison

Solutions from all three methods at $t = T$ alongside the exact solution, plus a quantitative error summary.

In [ ]:
# Interpolate FD solutions onto the common evaluation grid
from scipy.interpolate import interp1d

u_ex_ev     = interp1d(x_ex, u_ex_hist[-1], kind="linear",
                        bounds_error=False, fill_value=0.0)(x_ev)
u_im_ev     = interp1d(x_im, u_im_hist[-1], kind="linear",
                        bounds_error=False, fill_value=0.0)(x_ev)
u_pinn_ev   = U_pinn_snaps[-1]

err_ex_ev   = np.abs(u_ex_ev   - U_ex_pinn)
err_im_ev   = np.abs(u_im_ev   - U_ex_pinn)
err_pinn_ev = np.abs(u_pinn_ev - U_ex_pinn)

fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

labels  = ["Explicit", "Implicit (CN)", "PINN", "Exact"]
sols    = [u_ex_ev, u_im_ev, u_pinn_ev, U_ex_pinn]
colors  = ["steelblue", "seagreen", "mediumorchid", "black"]
errs    = [err_ex_ev, err_im_ev, err_pinn_ev]
e_lbl   = ["Explicit error", "Implicit error", "PINN error"]

for col, (lbl, sol, c) in enumerate(zip(labels, sols, colors)):
    ax = fig.add_subplot(gs[0, col])
    ax.plot(x_ev, sol, color=c, lw=2)
    ax.fill_between(x_ev, sol, alpha=0.2, color=c)
    ax.set_title(lbl, fontsize=11)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$u$")
    ax.set_ylim(-1.3, 1.3)
    ax.grid(alpha=0.3)

for col, (lbl, err) in enumerate(zip(e_lbl, errs)):
    ax = fig.add_subplot(gs[1, col])
    ax.plot(x_ev, err, "r-", lw=1.5)
    ax.set_title(f"{lbl}  (max {err.max():.1e})", fontsize=10)
    ax.set_xlabel("$x$")
    ax.set_ylabel("|error|")
    ax.grid(alpha=0.3)

# Overlay plot for direct comparison
ax_ov = fig.add_subplot(gs[1, 3])
ax_ov.plot(x_ev, U_ex_pinn,  "k-",  lw=2,   label="Exact")
ax_ov.plot(x_ev, u_ex_ev,    "b--", lw=1.5, label="Explicit")
ax_ov.plot(x_ev, u_im_ev,    "g:",  lw=1.5, label="Implicit")
ax_ov.plot(x_ev, u_pinn_ev,  "m-.", lw=1.5, label="PINN")
ax_ov.set_title("$t = T$ overlay", fontsize=10)
ax_ov.set_xlabel("$x$")
ax_ov.set_ylabel("$u$")
ax_ov.legend(fontsize=8)
ax_ov.grid(alpha=0.3)

fig.suptitle(f"Final Comparison at $t = T = {T_END}$ — Beam Equation",
             fontsize=13, fontweight="bold")
plt.show()

In [ ]:
# ---- Quantitative error summary ----
def l2_norm(f_arr, x_arr):
    return np.sqrt(np.trapezoid(f_arr**2, x_arr))


# Energy = ½ ∫ [u_t² + γ² u_xx²] dx — at t = 0:
#   u_t(x,0) = 0,  u_xx(x,0) = -(π/L)² sin(πx/L)
#   E₀ = ½ γ² (π/L)⁴ ∫₀ᴸ sin²(πx/L) dx = γ² π⁴ / (4 L³)
E0 = GAMMA**2 * np.pi**4 / (4 * L_DOM**3)

amp_exact = np.max(np.abs(U_ex_pinn))
amp_ex    = np.max(np.abs(u_ex_ev))
amp_im    = np.max(np.abs(u_im_ev))
amp_pinn  = np.max(np.abs(u_pinn_ev))

print("=" * 72)
print(f"{'Method':<22} {'Max error':>12} {'Mean error':>12} {'L² error':>12} {'Peak |u|':>10}")
print("-" * 72)
print(f"{'Exact':22} {'—':>12} {'—':>12} {'—':>12} {amp_exact:>10.4f}")
print(f"{'Explicit':22} {err_ex_ev.max():>12.3e} {err_ex_ev.mean():>12.3e} "
      f"{l2_norm(err_ex_ev, x_ev):>12.3e} {amp_ex:>10.4f}")
print(f"{'Implicit (CN-style)':22} {err_im_ev.max():>12.3e} {err_im_ev.mean():>12.3e} "
      f"{l2_norm(err_im_ev, x_ev):>12.3e} {amp_im:>10.4f}")
print(f"{'PINN':22} {err_pinn_ev.max():>12.3e} {err_pinn_ev.mean():>12.3e} "
      f"{l2_norm(err_pinn_ev, x_ev):>12.3e} {amp_pinn:>10.4f}")
print("=" * 72)
print(f"\nω₁ = γ(π/L)² = {OMEGA1:.4f},  period T₁ = {T_PER:.4f}")
print(f"Beam energy E₀ = γ²π⁴/(4L³) = {E0:.4f}  (conserved — no dissipation)")
print(f"Exact amplitude at t={T_END}: |cos(ω₁T)| = {abs(np.cos(OMEGA1 * T_END)):.4f}")

---

## Summary

### About the Beam Equation

The Euler–Bernoulli beam equation $u_{tt} + \gamma^2\,u_{xxxx} = 0$ is the **prototypical fourth-order linear hyperbolic PDE**. Key properties:

- **Bending stiffness** rather than tension: the restoring force comes from the beam's resistance to curvature ($u_{xxxx}$ vs $u_{xx}$ for the wave equation).
- **Quadratic dispersion** $\omega = \gamma k^2$: higher-frequency bending modes travel faster. The phase velocity $v_p = \gamma k$ grows linearly with wavenumber — the beam is *more dispersive* than the Airy equation ($\omega \propto k^3$ for fixed sign).
- **Energy conservation**: $E = \tfrac{1}{2}\int[u_t^2 + \gamma^2 u_{xx}^2]\,dx$ is constant (no dissipation). The bending energy $\tfrac{1}{2}\gamma^2\int u_{xx}^2\,dx$ and kinetic energy exchange but their sum is preserved.
- **Simply-supported BCs** ($u = u_{xx} = 0$ at both ends): the eigenfunctions are $\sin(n\pi x/L)$ with frequencies $\omega_n = \gamma(n\pi/L)^2$ growing as $n^2$.

### Method Comparison

| Aspect | Explicit (Biharmonic FD) | Implicit (CN Pentadiag.) | PINN |
|--------|------------------------|--------------------------|------|
| **Spatial stencil** | 5-point $[1,-4,6,-4,1]$ | Same, averaged over 3 time levels | Continuous (meshfree) |
| **Accuracy** | $\mathcal{O}(\Delta t^2,\,h^2)$ | $\mathcal{O}(\Delta t^2,\,h^2)$ | Depends on training |
| **Stability** | CFL: $\mu = \gamma^2\Delta t^2/h^4 \leq 1/4$ | Unconditional | Unconditional |
| **CFL severity** | $\Delta t \propto h^2$ (very restrictive) | N/A | N/A |
| **Linear system** | None (explicit) | Pentadiagonal (bandwidth 2) | None (gradient descent) |
| **BCs** | Ghost points from $u_{xx} = 0$ | Corner modifications to band matrix | Soft-constrained ($u = u_{xx} = 0$) |
| **Autograd passes** | — | — | 4 (for $u_{xxxx}$) + 2 (for $u_{tt}$) |
| **Best for** | Quick tests, coarse grids | Production — any time step | Inverse / parametric problems |

### Key Observations

- **CFL scales as $\Delta t \propto h^2$** for the explicit scheme — doubling the spatial resolution requires **four times** more time steps. With $N_x = 100$ this already requires $\sim\!10\,000$ time steps for $T = 1$. This makes implicit methods strongly preferable for fourth-order PDEs.
- The **pentadiagonal implicit scheme** absorbs the biharmonic operator into a banded LHS with bandwidth 2. Thanks to the simply-supported BCs, the corner diagonals change from $6\sigma$ to $5\sigma$ — a clean modification. The $\mathcal{O}(N)$ cost per solve makes this highly efficient.
- The **PINN** requires **four successive autograd passes** for $u_{xxxx}$ — the deepest derivative chain in this notebook collection. Each pass builds a deeper computational graph, increasing memory and compute cost. The Tanh activation is essential: ReLU would produce zero fourth derivatives.
- The **simply-supported BCs** ($u = 0$ *and* $u_{xx} = 0$) require the PINN to enforce **four boundary constraints** instead of the usual two (Dirichlet only). This is handled naturally in the soft-constraint loss but adds terms to $\mathcal{L}_{BC}$.